# Menyiapkan Dataset

In [1]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Tugas11-KopiNusantara").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df_tugas = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
df_tugas.printSchema()
df_tugas.show(5)
print("Total baris:", df_tugas.count())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 21:49:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows

Total baris: 2500


-------
## >> A. Feature Engineering

In [3]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

model_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx").fit(df_tugas)
model_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx").fit(df_tugas)
model_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx").fit(df_tugas)

df_idx = model_menu.transform(df_tugas)
df_idx = model_cabang.transform(df_idx)
df_idx = model_waktu.transform(df_idx)

assembler = VectorAssembler(
    inputCols=["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCol="features"
)
df_ml = assembler.transform(df_idx)
df_ml.select("features", "total_penjualan").show(5, truncate=False)

+-------------------------+------------------+
|features                 |total_penjualan   |
+-------------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|58157.94412015849 |
+-------------------------+------------------+
only showing top 5 rows



## >> B. Train/Test Split dan Pelatihan Model

In [4]:
from pyspark.ml.regression import LinearRegression

train, test = df_ml.randomSplit([0.75, 0.25], seed=101)
print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing  : {test.count()}")

lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model = lr.fit(train)

print("Koefisien:", model.coefficients)
print("Intercept:", model.intercept)

Jumlah data training : 1839
Jumlah data testing  : 661
Koefisien: [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept: -66903.17766485889


## >> C. Evaluasi Model

In [5]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = model.transform(test)

evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}")

RMSE : Rp 10,901
R²   : 0.9107


## >> D. Eksperimen Perbandingan Fitur

In [6]:
assembler_dasar = VectorAssembler(inputCols=["qty", "harga_satuan"], outputCol="features_dasar")

train_dasar = assembler_dasar.transform(train)
test_dasar = assembler_dasar.transform(test)

lr_dasar = LinearRegression(featuresCol="features_dasar", labelCol="total_penjualan")
model_dasar = lr_dasar.fit(train_dasar)

pred_dasar = model_dasar.transform(test_dasar)
rmse_dasar = evaluator_rmse.evaluate(pred_dasar)
r2_dasar = evaluator_r2.evaluate(pred_dasar)

print(f"R² model utama (5 fitur) : {r2:.4f}  | RMSE : Rp {rmse:,.0f}")
print(f"R² model kedua (2 fitur) : {r2_dasar:.4f}  | RMSE : Rp {rmse_dasar:,.0f}")
print(f"Selisih R²               : {r2 - r2_dasar:.4f}")

R² model utama (5 fitur) : 0.9107  | RMSE : Rp 10,901
R² model kedua (2 fitur) : 0.9108  | RMSE : Rp 10,896
Selisih R²               : -0.0001


## >> E. Prediksi Skenario Baru

In [7]:
skenario = [
    ("Kopi Susu", "Magelang", "Pagi", 3, 18000),
    ("Americano", "Yogyakarta", "Siang", 2, 22000),
    ("Roti Bakar", "Semarang", "Malam", 5, 15000),
]
df_skenario = spark.createDataFrame(
    skenario, ["nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan"]
)

df_skenario_idx = model_menu.transform(df_skenario)
df_skenario_idx = model_cabang.transform(df_skenario_idx)
df_skenario_idx = model_waktu.transform(df_skenario_idx)

df_skenario_ml = assembler.transform(df_skenario_idx)

hasil = model.transform(df_skenario_ml)
hasil.select("nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction").show(truncate=False)

+----------+----------+-----------+---+------------+-----------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction       |
+----------+----------+-----------+---+------------+-----------------+
|Kopi Susu |Magelang  |Pagi       |3  |18000       |53526.17351912391|
|Americano |Yogyakarta|Siang      |2  |22000       |44652.97009199053|
|Roti Bakar|Semarang  |Malam      |5  |15000       |89079.30253329039|
+----------+----------+-----------+---+------------+-----------------+

